# Agent Memory Toolkit – Function App (Remote Processor) Demo (async)

Async variant of `Demo_function_app.ipynb`. Wires `AsyncDurableFunctionProcessor` to
`AsyncCosmosMemoryClient` so the SDK only writes raw turns and the deployed sibling Azure Function
App handles all summarisation / fact-extraction asynchronously.

See `Demo_function_app.ipynb` for the full pattern explanation.

## 1. Setup

In [ ]:
import os, uuid
from dotenv import load_dotenv

load_dotenv(override=True)

from agent_memory_toolkit.aio import AsyncCosmosMemoryClient, AsyncDurableFunctionProcessor

print("Cosmos endpoint :", os.environ["COSMOS_DB_ENDPOINT"])
print("AI Foundry      :", os.environ["AI_FOUNDRY_ENDPOINT"])

## 2. Construct the async client with `AsyncDurableFunctionProcessor`

In [ ]:
memory = AsyncCosmosMemoryClient(
    cosmos_endpoint=os.environ["COSMOS_DB_ENDPOINT"],
    cosmos_key=os.environ.get("COSMOS_DB_KEY"),
    cosmos_database=os.environ.get("COSMOS_DB_DATABASE", "ai_memory"),
    cosmos_container=os.environ.get("COSMOS_DB_CONTAINER", "memories"),
    ai_foundry_endpoint=os.environ["AI_FOUNDRY_ENDPOINT"],
    chat_deployment_name=os.environ["AI_FOUNDRY_CHAT_DEPLOYMENT_NAME"],
    embedding_deployment_name=os.environ["AI_FOUNDRY_EMBEDDING_DEPLOYMENT_NAME"],
    processor=AsyncDurableFunctionProcessor(),
)
await memory.connect_cosmos()
print(f"Processor: {type(memory._processor).__name__}")

## 3. Write a conversation

In [ ]:
USER_ID   = f"fa-demo-{uuid.uuid4().hex[:8]}"
THREAD_ID = str(uuid.uuid4())
print(f"USER_ID   = {USER_ID}")
print(f"THREAD_ID = {THREAD_ID}")

transcript = [
    ("user",  "What's the weather like in Seattle this weekend?"),
    ("agent", "Around 55°F with partly cloudy skies on Saturday and light rain on Sunday."),
    ("user",  "Can you book me a weekend trip there? Flights under $300, hotel under $200."),
    ("agent", "I found a $275 Alaska Airlines round-trip and a $185/night hotel in Belltown."),
    ("user",  "Whenever you book a flight for me, always book an aisle seat — never window or middle."),
    ("agent", "Got it — aisle seats only."),
    ("user",  "For trip planning, my workflow is: first weather, then flights, then hotel."),
    ("agent", "Noted — weather → flights → hotel."),
    ("user",  "Never book me into anything between midnight and 6am unless I explicitly approve."),
    ("agent", "Understood — no overnight bookings without your approval."),
]
for role, content in transcript:
    await memory.add_cosmos(
        user_id=USER_ID,
        thread_id=THREAD_ID,
        role=role,
        content=content,
        memory_type="turn",
    )
    print(f"  wrote {role:>5}: {content[:60]}{'…' if len(content) > 60 else ''}")

## 4. `process_now()` is a no-op locally

In [ ]:
await memory.process_now(user_id=USER_ID, thread_id=THREAD_ID)
print("process_now() returned — no LLM call was made by the SDK.")

## 5. Wait for the Function App to produce a summary

In [ ]:
print("Polling Cosmos for the auto-generated summary…")
ok = await memory.process_now_and_wait(user_id=USER_ID, thread_id=THREAD_ID, timeout=120.0)
print(f"Summary available: {ok}")

## 6. Inspect what the Function App produced

In [ ]:
# All derived memories for this user, written by the function app (async).
import asyncio

async def _show():
    counts = {}
    for mt in ("summary", "fact", "episodic", "procedural", "user_summary"):
        docs = await memory.get_memories(user_id=USER_ID, memory_types=[mt])
        counts[mt] = len(docs)
        print(f"\n{mt.upper()}S ({len(docs)}):")
        for d in docs:
            print(f"  • [{d['id'][:32]}…] {d['content'][:90]}")
    return counts

print("Initial state:")
counts = await _show()

for attempt in range(6):
    if counts.get("fact", 0) > 0 or counts.get("procedural", 0) > 0:
        break
    print(f"\n... fact / procedural extraction still in flight (attempt {attempt+1}/6, sleeping 15s)")
    await asyncio.sleep(15)
    counts = {}
    for mt in ("summary", "fact", "episodic", "procedural", "user_summary"):
        docs = await memory.get_memories(user_id=USER_ID, memory_types=[mt])
        counts[mt] = len(docs)

print("\n=== Final state ===")
await _show()


## 7. Cleanup

In [ ]:
await memory.close()
print("Async client closed.")